# 从原始AF3输出聚合指标到summary CSV

读取每个complex的 metrics_summary.csv + confidences.json + summary_confidences.json，
聚合为每个方法的综合 summary CSV。

In [1]:
import os
import json
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser, FastMMCIFParser, Superimposer

In [2]:
BASE = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/predict/original_result'
METHODS = ['msa_notemplate', 'msa_template', 'nomsa_notemplate', 'nomsa_template']

AVAILABLE = [m for m in METHODS if os.path.isdir(f'{BASE}/{m}/output') and len(os.listdir(f'{BASE}/{m}/output')) > 0]
print(f'Available methods with data: {AVAILABLE}')

Available methods with data: ['msa_notemplate', 'msa_template', 'nomsa_notemplate', 'nomsa_template']


In [ ]:
os.makedirs('./summary', exist_ok=True)

for method in AVAILABLE:
    # 将方法名映射到notebook中使用的命名
    if method == 'msa_template':
        name = 'pro_msa_template-pep_nomsa_notemplate'
    elif method == 'msa_notemplate':
        name = 'pro_msa_notemplate-pep_nomsa_notemplate'
    elif method == 'nomsa_template':
        name = 'pro_nomsa_template-pep_nomsa_notemplate'
    elif method == 'nomsa_notemplate':
        name = 'pro_nomsa_notemplate-pep_nomsa_notemplate'
    else:
        name = method
    
    result = aggregate_method(method)
    if result is not None:
        result.to_csv(f'./summary/{name}.csv', index=False)
        print(f'{name}: {len(result)} rows, {result["native"].nunique()} complexes')
    else:
        print(f'{name}: no data')

In [ ]:
# 验证生成的summary文件
for f in sorted(os.listdir('./summary')):
    df = pd.read_csv(f'./summary/{f}')
    print(f'{f}: shape={df.shape}, columns={list(df.columns)}')